# Milestone 2: Data Cleaning and Preparation

Goal: build a reproducible pandas pipeline that cleans the 8 raw datasets, engineers features, and exports clean files to data/processed/.

In [2]:
import pandas as pd
from pathlib import Path
raw_path = Path("../data/raw")

sales = pd.read_csv(raw_path / "sales_transactions.csv")
customers = pd.read_csv(raw_path / "customers.csv")
stores = pd.read_csv(raw_path / "stores.csv")
products = pd.read_csv(raw_path / "products.csv")
promotions = pd.read_csv(raw_path / "promotions.csv")
cpi = pd.read_csv(raw_path / "knbs_cpi_inflation.csv")
demographics = pd.read_csv(raw_path / "knbs_county_demographics.csv")
holidays = pd.read_csv(raw_path / "kenya_public_holidays.csv")

print("Loaded all 8 files")


Loaded all 8 files


In [3]:
print("Shape:", sales.shape)
print()
print("Data types:")
print(sales.dtypes)
print()
print("Missing values per column:")
print(sales.isna().sum())
print()
print("Fully duplicated rows:", sales.duplicated().sum())
print("Duplicated transaction_id + product_id:", sales.duplicated(subset=["transaction_id", "product_id"]).sum())

Shape: (123927, 15)

Data types:
transaction_id             str
transaction_date           str
customer_id                str
store_id                   str
product_id                 str
quantity                 int64
unit_cost_kes          float64
unit_price_kes         float64
gross_sales_kes        float64
promo_id                   str
discount_amount_kes    float64
net_sales_kes          float64
cogs_kes               float64
gross_profit_kes       float64
payment_method             str
dtype: object

Missing values per column:
transaction_id            0
transaction_date          0
customer_id            6276
store_id                  0
product_id                0
quantity                  0
unit_cost_kes             0
unit_price_kes            0
gross_sales_kes           0
promo_id                  0
discount_amount_kes       0
net_sales_kes             0
cogs_kes                  0
gross_profit_kes          0
payment_method         2520
dtype: int64

Fully duplicated rows: 230

In [4]:
sales_clean = sales.copy()

# Convert the date from text to a real date
sales_clean["transaction_date"] = pd.to_datetime(sales_clean["transaction_date"], errors="coerce")

# Check the result
print("Date type:", sales_clean["transaction_date"].dtype)
print("Dates that failed to convert:", sales_clean["transaction_date"].isna().sum())
print("Earliest date:", sales_clean["transaction_date"].min())
print("Latest date:", sales_clean["transaction_date"].max())

Date type: datetime64[us]
Dates that failed to convert: 0
Earliest date: 2023-01-01 00:00:00
Latest date: 2024-12-31 00:00:00


In [5]:
print(sales_clean["payment_method"].value_counts(dropna=False))
print()
print(sales_clean["promo_id"].value_counts(dropna=False))
print()
print(sales_clean["store_id"].value_counts(dropna=False))

payment_method
M-Pesa    82876
Cash      26630
Card      11901
NaN        2520
Name: count, dtype: int64

promo_id
PROMO_NONE          82731
PROMO_XMAS23         5342
PROMO_XMAS24         5306
PROMO_SCHOOL24       5256
PROMO_SCHOOL23       5146
PROMO_EASTER23       5093
PROMO_MADARAKA23     5092
PROMO_EASTER24       4991
PROMO_MADARAKA24     4970
Name: count, dtype: int64

store_id
STR_008    12769
STR_002    12604
STR_004    12464
STR_006    12402
STR_005    12377
STR_003    12364
STR_010    12321
STR_001    12294
STR_009    12275
STR_007    12057
Name: count, dtype: int64


In [6]:
# --- Customer: tag guests ---
sales_clean["is_guest"] = sales_clean["customer_id"].isna()
sales_clean["customer_id"] = sales_clean["customer_id"].fillna("GUEST")

# --- Payment method: fill from other lines of the same transaction ---
missing_before = sales_clean["payment_method"].isna().sum()

known_payment = (
    sales_clean.dropna(subset=["payment_method"])
    .groupby("transaction_id")["payment_method"]
    .first()
)
sales_clean["payment_method"] = sales_clean["payment_method"].fillna(
    sales_clean["transaction_id"].map(known_payment)
)

missing_after_fill = sales_clean["payment_method"].isna().sum()

# Anything still missing becomes Unknown
sales_clean["payment_method"] = sales_clean["payment_method"].fillna("Unknown")

print("Guest rows:", sales_clean["is_guest"].sum())
print("Payment missing before:", missing_before)
print("Filled from same transaction:", missing_before - missing_after_fill)
print("Tagged as Unknown:", missing_after_fill)
print()
print(sales_clean.isna().sum())

Guest rows: 6276
Payment missing before: 2520
Filled from same transaction: 0
Tagged as Unknown: 2520

transaction_id         0
transaction_date       0
customer_id            0
store_id               0
product_id             0
quantity               0
unit_cost_kes          0
unit_price_kes         0
gross_sales_kes        0
promo_id               0
discount_amount_kes    0
net_sales_kes          0
cogs_kes               0
gross_profit_kes       0
payment_method         0
is_guest               0
dtype: int64


In [7]:
rows_before = len(sales_clean)
print("Duplicate rows found:", sales_clean.duplicated().sum())

sales_clean = sales_clean.drop_duplicates().reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after:", len(sales_clean))
print("Rows removed:", rows_before - len(sales_clean))

Duplicate rows found: 230
Rows before: 123927
Rows after: 123697
Rows removed: 230


In [8]:
print("Quantity <= 0:", (sales_clean["quantity"] <= 0).sum())
print("Unit price <= 0:", (sales_clean["unit_price_kes"] <= 0).sum())
print("Unit cost <= 0:", (sales_clean["unit_cost_kes"] <= 0).sum())
print("Unit price below unit cost:", (sales_clean["unit_price_kes"] < sales_clean["unit_cost_kes"]).sum())
print("Negative discount:", (sales_clean["discount_amount_kes"] < 0).sum())
print("Negative net sales:", (sales_clean["net_sales_kes"] < 0).sum())
print()

# Do the money columns add up?
gross_ok = (sales_clean["quantity"] * sales_clean["unit_price_kes"] - sales_clean["gross_sales_kes"]).abs() < 0.01
net_ok = (sales_clean["gross_sales_kes"] - sales_clean["discount_amount_kes"] - sales_clean["net_sales_kes"]).abs() < 0.01
profit_ok = (sales_clean["net_sales_kes"] - sales_clean["cogs_kes"] - sales_clean["gross_profit_kes"]).abs() < 0.01

print("gross_sales mismatches:", (~gross_ok).sum())
print("net_sales mismatches:", (~net_ok).sum())
print("gross_profit mismatches:", (~profit_ok).sum())
print()
print(sales_clean[["quantity", "unit_price_kes", "net_sales_kes"]].describe())

Quantity <= 0: 0
Unit price <= 0: 0
Unit cost <= 0: 0
Unit price below unit cost: 0
Negative discount: 0
Negative net sales: 0

gross_sales mismatches: 0
net_sales mismatches: 0
gross_profit mismatches: 0

            quantity  unit_price_kes  net_sales_kes
count  123697.000000   123697.000000  123697.000000
mean        1.764683      196.090811     330.177020
std         1.044177      149.635470     354.490189
min         1.000000       50.000000      40.000000
25%         1.000000      110.000000     130.000000
50%         1.000000      160.000000     204.000000
75%         2.000000      210.000000     390.000000
max         5.000000      680.000000    3400.000000
